# 5 — Blueprint et soumission Kaggle (CRISP-DM Phase 6)

Ce notebook joue **deux rôles** :

1. **Synthèse exécutive** (« rapport à la direction Inved Corp ») — récapitulatif business → modèle final → évaluation → déploiement → risques.
2. **Soumission Kaggle** — applique le pipeline du modèle champion à `data/test.csv` et génère `submission.csv`.

**Pré-requis** : avoir exécuté `4_evaluation.ipynb` au moins une fois (lit `results/winning_model.json`).

---

## Sommaire du blueprint

- Introduction de l'évaluation (pas un résumé des étapes 1-2)
- Stratégie de déploiement (CI/CD, versioning, environnements de dev/test/prod)
- *Operate* — mise à disposition aux end-users, mode d'exécution (real-time / batch)
- Monitoring sur trois axes :
    - **Mathématique** — RMSLE en production, PSI/KS data drift, prediction drift, SHAP stability
    - **Business** — KPIs ML Canvas (temps gagné, taux d'override manuel, conversion 90j)
    - **IT** — latence p50/p95/p99, error rate, logs structurés, événements de déploiement
- Feedback quantitatif/qualitatif (automatique + manuel via CRM)
- Cas d'école transverse : *gentrification* (Ames / campus ISU) — touche les trois axes simultanément
- Risques et limites du modèle champion

## 6.1 Setup et identification du champion

In [ ]:
%load_ext autoreload
%autoreload 2
%run 2_data_prep.ipynb

In [ ]:
winner_path = RESULTS_DIR / 'winning_model.json'
if not winner_path.exists():
    raise RuntimeError(
        "results/winning_model.json absent. Exécuter 4_evaluation.ipynb d'abord."
    )

winner = json.loads(winner_path.read_text())
print(f"Modèle champion désigné par 4_evaluation : {winner['model']}  (famille : {winner['family']})")
print(f"  Holdout RMSLE : {winner['holdout_rmsle']:.4f}")
print(f"  Params        : {winner.get('params')}")

## 6.2 Reconstruction et réentraînement du champion sur **toutes** les données d'entraînement

Pour la soumission Kaggle on entraîne sur 100 % des données disponibles (`X` complet, pas seulement `X_train` qui exclut 20 %). Le préprocesseur reste construit selon les règles définies dans `2_data_prep.ipynb`.

Le *dispatcher* ci-dessous mappe chaque nom de modèle vers la fonction qui le reconstruit. Quand on ajoutera LightGBM ou CatBoost, il suffira de rallonger ce dispatcher.

In [ ]:
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV, ElasticNetCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, StackingRegressor
from xgboost import XGBRegressor


def _scaled(model):
    return Pipeline([('preprocessor', preprocessor_scaled), ('model', model)])

def _encoded(model):
    return Pipeline([('preprocessor', preprocessor_encoded), ('model', model)])

def _native(model):
    return Pipeline([('preprocessor', preprocessor_native), ('model', model)])


def build_champion(model_name: str, params: dict | None):
    p = params or {}

    if model_name == 'OLS':
        return _scaled(LinearRegression())
    if model_name == 'Ridge':
        return _scaled(RidgeCV(alphas=np.logspace(-2, 2, 20), cv=5))
    if model_name == 'Lasso':
        return _scaled(LassoCV(alphas=np.logspace(-4, 2, 20), cv=5, max_iter=50_000, random_state=RANDOM_STATE))
    if model_name == 'ElasticNet':
        return _scaled(ElasticNetCV(l1_ratio=[.1, .5, .7, .9, .95, .99, 1],
                                    alphas=np.logspace(-4, 2, 20), cv=5, max_iter=50_000, random_state=RANDOM_STATE))
    if model_name == 'KNN':
        return _scaled(KNeighborsRegressor(n_neighbors=int(p.get('k', 9))))
    if model_name == 'MLPRegressor':
        return _scaled(MLPRegressor(hidden_layer_sizes=(64,), activation='relu', solver='adam',
                                    max_iter=1500, early_stopping=True, validation_fraction=0.15,
                                    random_state=RANDOM_STATE))

    if model_name == 'DecisionTree':
        return _encoded(DecisionTreeRegressor(max_depth=int(p.get('max_depth', 6)), random_state=RANDOM_STATE))
    if model_name == 'RandomForest':
        return _encoded(RandomForestRegressor(n_estimators=int(p.get('n_estimators', 300)),
                                              random_state=RANDOM_STATE, n_jobs=-1))
    if model_name == 'GradientBoosting':
        return _encoded(GradientBoostingRegressor(
            n_estimators=int(p.get('n_estimators', 300)),
            learning_rate=float(p.get('learning_rate', 0.05)),
            max_depth=int(p.get('max_depth', 3)),
            random_state=RANDOM_STATE,
        ))
    if model_name == 'AdaBoost':
        return _encoded(AdaBoostRegressor(n_estimators=int(p.get('n_estimators', 200)), random_state=RANDOM_STATE))

    if model_name in {'XGBoost_native', 'XGBoost_onehot', 'XGBoost_tuned'}:
        defaults = dict(n_estimators=500, learning_rate=0.05, max_depth=4)
        cfg = {**defaults, **{k: v for k, v in p.items() if k in {
            'n_estimators', 'learning_rate', 'max_depth',
            'min_child_weight', 'subsample', 'colsample_bytree',
            'reg_alpha', 'reg_lambda',
        }}}
        # XGBoost_onehot used preprocessor_encoded; the others use preprocessor_native
        prep = preprocessor_encoded if model_name == 'XGBoost_onehot' else preprocessor_native
        kwargs = dict(random_state=RANDOM_STATE, n_jobs=-1, verbosity=0, tree_method='hist')
        if prep is preprocessor_native:
            kwargs['enable_categorical'] = True
        return Pipeline([('preprocessor', prep), ('model', XGBRegressor(**cfg, **kwargs))])

    if model_name == 'Stacking':
        lasso_sub = _scaled(LassoCV(alphas=np.logspace(-4, 2, 20), cv=5, max_iter=50_000, random_state=RANDOM_STATE))
        gb_sub    = _encoded(GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE))
        xgb_sub   = _native(XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=4,
                                         enable_categorical=True, tree_method='hist',
                                         random_state=RANDOM_STATE, n_jobs=-1, verbosity=0))
        return StackingRegressor(
            estimators=[('lasso', lasso_sub), ('gbr', gb_sub), ('xgb', xgb_sub)],
            final_estimator=RidgeCV(alphas=np.logspace(-2, 2, 10)),
            cv=5, n_jobs=-1, passthrough=False,
        )

    raise ValueError(f"Modèle inconnu dans le dispatcher : {model_name}")


champion = build_champion(winner['model'], winner.get('params'))
print(f"Champion reconstruit : {type(champion).__name__}")

## 6.3 Entraînement final sur 100 % des données

`X` et `y_log` sont définis par `2_data_prep.ipynb` (avant le `train_test_split`). On les utilise tels quels pour bénéficier de toutes les observations disponibles.

In [ ]:
t0 = time.time()
champion.fit(X, y_log)
fit_s = time.time() - t0
print(f"Champion entraîné sur {len(X)} observations en {fit_s:.1f}s")

## 6.4 Application au jeu de test Kaggle

On charge `data/test.csv` (le **vrai** test set Kaggle, sans `SalePrice`), on applique exactement les mêmes drops de colonnes que sur le train (`Id`, `SalePrice*`), puis on prédit en espace log avant retransformation `expm1`.

In [ ]:
df_test = pd.read_csv('./data/test.csv', sep=',')
test_ids = df_test['Id']
print(f"Test Kaggle : {df_test.shape}")

X_test_kaggle = df_test.drop(columns=['SalePrice', 'SalePrice_log', 'Id'], errors='ignore')
# Align column order with training X (defensive — le preprocessor le ferait aussi)
X_test_kaggle = X_test_kaggle[X.columns]

preds_log = champion.predict(X_test_kaggle)
preds_dollars = np.expm1(preds_log)

print(f"Statistiques des prédictions ($) :")
print(f"  min: {preds_dollars.min():,.0f}")
print(f"  mean: {preds_dollars.mean():,.0f}")
print(f"  max: {preds_dollars.max():,.0f}")
print(f"  nb NaN: {int(np.isnan(preds_dollars).sum())}")

assert not np.isnan(preds_dollars).any(), "Prédictions contiennent des NaN — modèle ou prétraitement défaillant"
assert (preds_dollars > 0).all(), "Prédictions <= 0 — anomalie"
assert len(preds_dollars) == len(df_test), "Longueur de prédictions ≠ longueur du test set"

submission = pd.DataFrame({'Id': test_ids, 'SalePrice': preds_dollars})
submission.to_csv('submission.csv', index=False)
print(f"\nSoumission écrite : submission.csv ({len(submission)} lignes)")
display(submission.head())

## 6.5 Notes pour la soutenance / livrable

À développer collectivement (placeholder volontaire) :

- **Pourquoi ce champion gagne** — diversité d'erreurs entre Lasso/GBR/XGB exploitée par le RidgeCV méta-modèle (cf. diagnostic dans `3d_stacking.ipynb` §4.3).
- **Coût opérationnel** — Stacking entraîne 4× plus longtemps qu'un modèle seul. Si la dégradation RMSLE d'XGBoost tuné est < 1 %, plaider pour XGBoost en prod (un seul artefact, une seule erreur potentielle à debugger).
- **Monitoring trois axes** — math (RMSLE prod), business (Canvas KPIs), IT (latence p95 < 5s — Canvas SLA).
- **Limites du champion** — drift du marché (fenêtre glissante mensuelle prévue), gentrification (cas d'école transverse), opacité Stacking pour les consultants.
- **Plan T13/T14/T15** — MLflow (registry + serve), POC Streamlit, persistance joblib.